# 08. Paoay: satellite inputs

Prepare Sentinel-2 imagery and matching Dynamic World probabilities for one four-month period. Verified existing files are reused; only missing files can be downloaded. The model and existing archives are untouched.


## 1. Setup and dates

Run 07 first. January–April 2024 remains the historical baseline. Its clear coverage does not establish crop-cycle compatibility or verify January field conditions. Record the intended comparison and review the reference checklist before changing the period. Each source has its own `download_missing` / `reuse` mode.


In [ ]:
from pathlib import Path
import json
import sys

repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import imagery, dynamic_world
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress

settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()

period_start = "2024-01-01"  # January–April 2024; end exclusive: 2024-05-01.
imagery_mode = "reuse"      # Download only missing files; use "reuse" for local-only inspection.
dynamic_world_mode = "reuse"  # Use "reuse" for local-only checks.
period_purpose = "historical_baseline"  # This is not a January field-verified map.
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
paoay_root = report_folder.parent / "paoay"
reference_pointer = paoay_root / "references/reference_stage.json"
imagery_root = paoay_root / "sentinel2"
imagery_pointer = imagery_root / "imagery_stage.json"
dynamic_world_root = paoay_root / "dynamic_world"
dynamic_world_pointer = dynamic_world_root / "dynamic_world_stage.json"
# Invalidate both selections before any work so a failed run cannot select stale inputs.
write_stage_pointer(imagery_pointer, status="pending", reason="Notebook 08 satellite inputs are incomplete")
write_stage_pointer(dynamic_world_pointer, status="pending", reason="Notebook 08 satellite inputs are incomplete")
if imagery_mode not in {"reuse", "download_missing"} or dynamic_world_mode not in {"reuse", "download_missing"}:
    raise ValueError("Choose reuse or download_missing for each satellite source")
stages = ["Plan the period and grid", "Check local imagery", "Prepare matching composites", "Verify imagery handoff",
          "Check local Dynamic World", "Prepare probability inputs", "Verify Dynamic World handoff", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook(
    "08_paoay_satellite_inputs", stages, report_folder.parent / "progress",
    {"start": period_start, "purpose": period_purpose, "sentinel2_mode": imagery_mode, "dynamic_world_mode": dynamic_world_mode},
)


## 2. Plan the period and grid

Use the verified study boundary, snapped outward to a 10 m grid in its local projected CRS. No extra parcel buffer is added. The rectangular TIFFs include a margin outside the boundary; coverage statistics use only pixel centres inside it.


In [ ]:
pipeline_progress.begin_stage("Plan the period and grid")
reference_handoff = resolve_stage_pointer(reference_pointer)
request = imagery.prepare_request(reference_handoff, period_start, imagery_root)
print("Period:", request["period"]["start"], "to", request["period"]["end"], "(end exclusive)")
print("Grid CRS:", request["settings"]["grid"]["crs"])
print("Pixel size: 10 m; composites: 8")
print("Request folder:", request["folder"])
show_table(request["period"]["intervals"], title="Half-month inputs")
pipeline_progress.end_stage()


## 3. Check local imagery

Only files with matching geometry, dates and processing receipts are reused. Legacy Cloud Score+ TIFFs and external training archives are left untouched. Changed or incomplete cached pairs require inspection and are never silently overwritten.


In [ ]:
pipeline_progress.begin_stage("Check local imagery")
local_status = imagery.cache_status(request)
show_table(local_status, title="Matching local imagery")
missing_composites = sum(row["status"] == "missing" for row in local_status)
print("Missing composites:", missing_composites)
pipeline_progress.end_stage()


## 4. Prepare matching composites

The shared training recipe uses Sentinel-2 SR Harmonized, exact cloud-probability matches, probability < 40, SCL/edge masks and all six valid bands. It mosaics observations from the same date before each half-month median. B5/B11 use bilinear resampling; masks retain nearest-neighbour handling.

Each TIFF stores **B3, B4, B5, B8, B11, B2, clear_count**. Reflectance stays unscaled (apply 0.0001 later); clear counts are never scaled. Empty bins retain nodata and zero counts. No gap filling or quality thresholds are relaxed.


In [ ]:
pipeline_progress.begin_stage("Prepare matching composites")
earth_engine = None
if imagery_mode == "download_missing" and missing_composites:
    project = local_settings.get("earth_engine_project", "").strip()
    if not project or project == "YOUR_EARTH_ENGINE_PROJECT":
        raise ValueError("Set your Earth Engine project in settings.local.json before downloading")
    from crop_pipeline.notebook_setup import configure_certificates
    configure_certificates()
    import ee
    try:
        ee.Initialize(project=project)
    except ee.EEException:
        ee.Authenticate()
        ee.Initialize(project=project)
    earth_engine = ee

def composite_progress(items):
    return pipeline_progress.track(items, "Prepare Sentinel-2 composites", unit="composite")

imagery_handoff = imagery.collect(request, mode=imagery_mode, earth_engine=earth_engine, iterator=composite_progress)
show_table(imagery.observation_report(imagery_handoff), title="Sentinel-2 observation audit")
pipeline_progress.end_stage()


## 5. Verify and save the handoff

A successful handoff requires all eight matching, checksum-verified raster/receipt pairs. An empty composite is a completed retrieval with insufficient observations, not a usable crop input. Later feature/model gates decide usable pixels.


In [ ]:
pipeline_progress.begin_stage("Verify imagery handoff")
image_bundle = imagery.load_imagery_bundle(imagery_handoff)
quality_rows = []
for interval in request["period"]["intervals"]:
    record = imagery.cached_interval(request, interval)
    quality_rows.append({"interval": interval["name"], **record["quality"]})
show_table(quality_rows, title="Clear observations inside the study boundary")
summary = image_bundle["manifest"]["summary"]
print("ROI pixels:", summary["roi_pixels"])
print("Pixels clear in all eight bins:", summary["all_eight_bins_clear_pixels"])
print("All-eight-bin clear coverage:", f"{summary['all_eight_bins_clear_fraction']:.1%}")
print("Empty half-month bins:", summary["empty_bins"])
pipeline_progress.end_stage()


## 6. Check matching Dynamic World inputs

Inherit the exact verified Sentinel-2 dates, grid and boundary from this run. Dynamic World predicts broad cover, not crop species or ground truth. A failed second source leaves both stage selections pending.


In [ ]:
pipeline_progress.begin_stage("Check local Dynamic World")
world_request = dynamic_world.prepare_request(imagery_handoff, dynamic_world_root)
world_status = dynamic_world.cache_status(world_request)
show_table(world_status, title="Matching local Dynamic World inputs")
missing_world_composites = sum(row["status"] == "missing" for row in world_status)
print("Missing Dynamic World composites:", missing_world_composites)
pipeline_progress.end_stage()


## 7. Prepare broad-cover probabilities

Keep provider masks, all nine probabilities and daily observation counts. Reuse matching pairs; never replace corrupt or unpaired saved files automatically. One progress bar covers this source.


In [ ]:
pipeline_progress.begin_stage("Prepare probability inputs")
if dynamic_world_mode == "download_missing" and missing_world_composites and earth_engine is None:
    project = local_settings.get("earth_engine_project", "").strip()
    if not project or project == "YOUR_EARTH_ENGINE_PROJECT":
        raise ValueError("Set your Earth Engine project in settings.local.json before downloading")
    from crop_pipeline.notebook_setup import configure_certificates
    configure_certificates()
    import ee
    try:
        ee.Initialize(project=project)
    except ee.EEException:
        ee.Authenticate()
        ee.Initialize(project=project)
    earth_engine = ee

def composite_progress(items):
    return pipeline_progress.track(items, "Prepare Dynamic World inputs", unit="composite")

dynamic_world_handoff = dynamic_world.collect(
    world_request, mode=dynamic_world_mode, earth_engine=earth_engine, iterator=composite_progress,
)
pipeline_progress.end_stage()


## 8. Verify both sources and publish

Check probabilities, counts, dates, grid and file checksums. Empty bins remain explicit. Retrieval coverage is not crop accuracy; downstream feature-quality checks remain unchanged.


In [ ]:
pipeline_progress.begin_stage("Verify Dynamic World handoff")
world_bundle = dynamic_world.load_dynamic_world_bundle(dynamic_world_handoff)
coverage_rows = []
for interval in world_request["period"]["intervals"]:
    record = dynamic_world.cached_interval(world_request, interval)
    coverage_rows.append({"interval": interval["name"], "available_dates": len(record["available_dates"]), **record["quality"]})
show_table(coverage_rows, title="Dynamic World coverage inside the study boundary")
summary = world_bundle["manifest"]["summary"]
print("ROI pixels:", summary["roi_pixels"])
print("Pixels observed in at least one bin:", summary["any_bin_observed_pixels"])
print("Pixels observed in all eight bins:", summary["all_eight_bins_observed_pixels"])
print("Empty half-month bins:", summary["empty_bins"])
imagery.load_imagery_bundle(imagery_handoff)
write_stage_pointer(imagery_pointer, imagery_handoff)
write_stage_pointer(dynamic_world_pointer, dynamic_world_handoff)
pipeline_progress.end_stage()


## 9. Summary

The two existing stage-pointer formats remain compatible with saved downstream artifacts. Next, run notebook 09 for exact model features and fixed-model predictions. Do not rerun the archived standalone Dynamic World notebook.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Sentinel-2 handoff:", imagery_handoff)
print("Dynamic World handoff:", dynamic_world_handoff)
print("Period:", request["period"]["start"], "to", request["period"]["end"], "(end exclusive)")
print("Purpose:", period_purpose)
print("Crop-cycle alignment: unverified; reference notes are not dated ground truth.")
print("Dynamic World attribution:", dynamic_world.ATTRIBUTION)
print("Next: notebook 09 applies the saved model. Nothing starts automatically.")
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
